<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/12-scikit-learn.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 12 — scikit-learn: The Interface, Not the Algorithms

Companion to [the chapter](https://www.ai.biz/books/python-primer/scikit-learn/).

This notebook teaches the **API**. It says nothing about how the models work.
The point is that you can be correct without knowing that yet.


In [ ]:
import numpy as np
import pandas as pd
import sklearn
print('sklearn', sklearn.__version__)


## 0. A small, deliberately awkward dataset

Numeric columns with gaps, categorical columns, and an ID that must never reach the model.


In [ ]:
rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    'customer_id': [f'C{i:05d}' for i in range(n)],
    'age': rng.normal(42, 12, n).round(),
    'income': rng.lognormal(10.5, 0.6, n).round(-2),
    'city': rng.choice(['delhi', 'tokyo', 'sydney', 'paris'], n, p=[.4,.3,.2,.1]),
    'region': rng.choice(['Asia', 'Europe', 'Americas'], n, p=[.5,.3,.2]),
})
# a signal to find, plus noise
logit = (df.age - 42) / 12 * 0.8 + (np.log(df.income) - 10.5) * 1.1 + rng.normal(0, 1, n)
df['churned'] = (logit > 0.5).astype(int)

# real data has gaps
df.loc[rng.choice(n, 150, replace=False), 'age'] = np.nan
df.loc[rng.choice(n, 90, replace=False), 'city'] = None

print(df.shape, '| churn rate', df.churned.mean().round(3))
df.head()


## 1. The estimator contract

Three unrelated algorithms, one loop. That is the whole design insight.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X = df[['age', 'income']].fillna(df[['age','income']].median())  # crude, fixed properly below
y = df.churned
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

for model in [LogisticRegression(max_iter=1000), RandomForestClassifier(random_state=0), SVC()]:
    model.fit(Xtr, ytr)
    print(f'{type(model).__name__:<24} {model.score(Xte, yte):.3f}')


In [ ]:
# Convention: hyperparameters in the constructor, learned state ends with _
m = RandomForestClassifier(n_estimators=50, random_state=0)
print('before fit, asked for n_estimators =', m.n_estimators)
try:
    m.feature_importances_
except Exception as e:
    print('before fit, feature_importances_ ->', type(e).__name__)
m.fit(Xtr, ytr)
print('after fit, feature_importances_ ->', m.feature_importances_.round(3))


## 2. Leakage, demonstrated

Scaling before splitting lets test-set statistics into training. The effect is small,
silent, and always in the flattering direction.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: scaler sees every row before the folds are made
X_leaked = StandardScaler().fit_transform(X)
leaked = cross_val_score(LogisticRegression(max_iter=1000), X_leaked, y, cv=cv, scoring='roc_auc')

# RIGHT: the scaler refits inside every fold
pipe = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
clean = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

print(f'leaked : {leaked.mean():.5f}')
print(f'honest : {clean.mean():.5f}')
print(f'gap    : {leaked.mean() - clean.mean():+.5f}  <- small here, and always flattering')


The gap is tiny on clean numeric data. It becomes large with imputation,
target encoding, or feature selection. The point is that you cannot see it,
so you must make it structurally impossible.


## 3. ColumnTransformer: different columns, different treatment


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric = ['age', 'income']
categorical = ['city', 'region']

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('encode', OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False))]), categorical),
], remainder='drop')   # customer_id is silently dropped. Deliberately.

preprocess.set_output(transform='pandas')   # keep column names
out = preprocess.fit_transform(df)
print(out.shape)
out.head(3)


In [ ]:
# handle_unknown='ignore' is what keeps production alive
unseen = pd.DataFrame({'age':[30.0],'income':[50000.0],'city':['lagos'],'region':['1']})
print('unseen city encodes to all zeros, no exception:')
print(preprocess.transform(unseen).filter(like='city'))


## 4. The full pipeline, cross-validated honestly


In [ ]:
from sklearn.model_selection import cross_validate

full = Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=1000))])

features = ['age','income','city','region']
Xtr, Xte, ytr, yte = train_test_split(df[features], y, test_size=0.2, stratify=y, random_state=42)

scores = cross_validate(full, Xtr, ytr, cv=cv, scoring=['roc_auc','average_precision'],
                        return_train_score=True)

for k in ['train_roc_auc','test_roc_auc','test_average_precision']:
    print(f'{k:<26} {scores[k].mean():.3f} ± {scores[k].std():.3f}')
print()
print('train well above test would mean overfitting; close together is healthy')


## 5. Always beat a dumb baseline

Three lines. It has saved more projects than any modelling technique.


In [ ]:
from sklearn.dummy import DummyClassifier
base = cross_val_score(DummyClassifier(strategy='prior'), Xtr, ytr, cv=cv, scoring='roc_auc')
print(f'baseline AUC : {base.mean():.3f}   <- coin flip, by construction')
print(f'model AUC    : {scores["test_roc_auc"].mean():.3f}')


## 6. Hyperparameter search with double underscores


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

search = RandomizedSearchCV(
    full,
    {'model__C': loguniform(1e-3, 1e2),
     'prep__num__impute__strategy': ['median', 'mean']},
    n_iter=12, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42,
)
search.fit(Xtr, ytr)
print('best params:', search.best_params_)
print(f'best cv AUC: {search.best_score_:.3f}')


## 7. A custom transformer that follows the conventions


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class ClipOutliers(BaseEstimator, TransformerMixin):
    '''Clip to percentiles LEARNED FROM THE TRAINING FOLD ONLY.'''
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower      # store unchanged; sklearn clones by reading these
        self.upper = upper
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        return self             # always
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)

clip_pipe = Pipeline([('clip', ClipOutliers()), ('scale', StandardScaler()),
                      ('model', LogisticRegression(max_iter=1000))])
print('works inside CV:', cross_val_score(clip_pipe, X, y, cv=cv, scoring='roc_auc').mean().round(3))


## 8. The test set, touched once, at the end


In [ ]:
final = search.best_estimator_
from sklearn.metrics import roc_auc_score, classification_report

proba = final.predict_proba(Xte)[:, 1]
print(f'TEST AUC: {roc_auc_score(yte, proba):.3f}')
print()
print(classification_report(yte, final.predict(Xte), digits=3))


## 9. Persist the whole pipeline, with its provenance


In [ ]:
import joblib, datetime, tempfile, os

artifact = {
    'pipeline': final,
    'sklearn_version': sklearn.__version__,
    'trained_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'features': features,
    'cv_score': search.best_score_,
}
path = os.path.join(tempfile.gettempdir(), 'model-v1.joblib')
joblib.dump(artifact, path)

loaded = joblib.load(path)
print('reloaded, trained at', loaded['trained_at'])
print('same predictions:', np.allclose(loaded['pipeline'].predict_proba(Xte)[:,1], proba))


## Try it yourself

1. Set `remainder='passthrough'` on the ColumnTransformer and watch it fail on `customer_id`. That failure is the guard working.
2. Swap `LogisticRegression` for `RandomForestClassifier` by changing one line. Nothing else should need editing.
3. Build a leakage demo with `SimpleImputer` fitted before the split. The gap will be larger than the scaling one.
4. Add a `SelectKBest` step *outside* a pipeline and then inside one, and compare the cross-validated scores.
5. Replace `StratifiedKFold` with `TimeSeriesSplit` and think about which of these features would exist at prediction time.
